# VisionAssist Phase 11b — Condition-balanced replay

This notebook runs a short recovery experiment from the promoted Phase 10 adapter. It rebuilds a 6,000-record validation-driven selection with exact per-task condition quotas, trains for at most 150 optimizer steps at `1e-5`, and evaluates validation before any frozen-test decision.

In [ ]:
#@title 1. Settings — run before importing Torch
import os
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/moshiur00/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")
DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
PILOT_RUN_ID = "qwen25vl3b_qlora_pilot_v1"
BALANCED_RUN_ID = "qwen25vl3b_qlora_balanced_replay_v1"
SELECTION_CONFIG = PROJECT_ROOT / "configs/training/phase11b_balanced_replay.yaml"
TRAINING_CONFIG = PROJECT_ROOT / "configs/training/qwen25vl3b_qlora_balanced_replay.yaml"

In [ ]:
#@title 2. Mount Drive and verify Phase 10 artifacts
from google.colab import drive
drive.mount("/content/drive")
DRIVE_PILOT = DRIVE_ROOT / "outputs/training" / PILOT_RUN_ID
DRIVE_PILOT_EVAL = DRIVE_ROOT / "outputs/post_training/qwen25vl3b_pilot_best"
required = [DRIVE_DATA_ARCHIVE, DRIVE_PILOT / "final_adapter/adapter_model.safetensors", DRIVE_PILOT_EVAL / "validation/predictions.jsonl", DRIVE_PILOT_EVAL / "validation/evaluation/metrics.json"]
for path in required: print(path, path.exists())
assert all(path.exists() for path in required)

In [ ]:
#@title 3. Clone/update repository and install dependencies
import shutil, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(["uv", "sync", "--extra", "training", "--extra", "dev"], cwd=PROJECT_ROOT, check=True)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())

In [ ]:
#@title 4. Verify GPU
import torch
assert torch.cuda.is_available(), "Select a GPU runtime."
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(properties.total_memory / 1024**3, 2))
print("BF16:", torch.cuda.is_bf16_supported())
subprocess.run(["nvidia-smi"], check=False)

## A. Restore data and promoted artifacts

In [ ]:
#@title 5. Restore prepared data and Phase 10 outputs
import tarfile
data_files = [PROJECT_ROOT / "data/raw/visa", PROJECT_ROOT / "data/processed/visa_instructions/train.jsonl", PROJECT_ROOT / "data/processed/visa_instructions/validation.jsonl", PROJECT_ROOT / "data/processed/visa_instructions/test.jsonl", PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"]
if not all(path.exists() for path in data_files):
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive: archive.extractall(PROJECT_ROOT, filter="data")
assert all(path.exists() for path in data_files)
LOCAL_PILOT = PROJECT_ROOT / "outputs/training" / PILOT_RUN_ID
LOCAL_PILOT_EVAL = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_pilot_best"
shutil.copytree(DRIVE_PILOT, LOCAL_PILOT, dirs_exist_ok=True)
shutil.copytree(DRIVE_PILOT_EVAL, LOCAL_PILOT_EVAL, dirs_exist_ok=True)
print("Prepared data and Phase 10 reference restored.")

In [ ]:
#@title 6. Normalize image paths and refresh benchmark hash
import hashlib, json
from pathlib import PurePosixPath
MARKER = ("data", "raw", "visa")
def normalize_path(value):
    parts = PurePosixPath(str(value).replace("\\", "/")).parts
    lowered = tuple(part.lower() for part in parts)
    for index in range(len(parts) - 2):
        if lowered[index:index + 3] == MARKER: return PurePosixPath(*parts[index:]).as_posix()
    candidate = PurePosixPath(*parts)
    if not candidate.is_absolute() and ".." not in candidate.parts: return candidate.as_posix()
    raise ValueError(value)
def normalize_jsonl(path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        row = json.loads(line)
        for message in row.get("messages", []):
            if message.get("role") != "user": continue
            for item in message.get("content", []):
                if item.get("type") == "image": item["image"] = normalize_path(item["image"])
        rows.append(row)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
    temporary.replace(path)
instruction_root = PROJECT_ROOT / "data/processed/visa_instructions"
for split in ("train", "validation", "test"): normalize_jsonl(instruction_root / f"{split}.jsonl")
benchmark = PROJECT_ROOT / "data/benchmarks/visa_baseline_v1/benchmark.jsonl"
normalize_jsonl(benchmark)
benchmark_hash = hashlib.sha256(benchmark.read_bytes()).hexdigest()
benchmark_manifest_path = benchmark.parent / "benchmark_manifest.json"
benchmark_manifest = json.loads(benchmark_manifest_path.read_text(encoding="utf-8")); benchmark_manifest["benchmark_sha256"] = benchmark_hash
benchmark_manifest_path.write_text(json.dumps(benchmark_manifest, indent=2) + "\n", encoding="utf-8")
(benchmark.parent / "benchmark_sha256.txt").write_text(benchmark_hash + "\n", encoding="utf-8")

In [ ]:
#@title 7. Run Phase 11b regression tests
subprocess.run(["uv", "run", "pytest", "tests/test_phase8_training.py", "tests/test_phase11_hard_examples.py", "tests/test_phase11_failure_analysis.py"], cwd=PROJECT_ROOT, check=True)

## B. Rebuild and audit condition-balanced replay

In [ ]:
#@title 8. Build the exact 6,000-record Phase 11b selection
import yaml
subprocess.run(["uv", "run", "visionassist", "select-hard-examples", "--config", str(SELECTION_CONFIG)], cwd=PROJECT_ROOT, check=True)
selection_config = yaml.safe_load(SELECTION_CONFIG.read_text(encoding="utf-8"))
selection_manifest = json.loads((PROJECT_ROOT / selection_config["manifest_path"]).read_text(encoding="utf-8"))
print(json.dumps(selection_manifest, indent=2))
assert selection_manifest["records"] == 6000
assert selection_manifest["unique_instruction_ids"] == 6000
assert selection_manifest["instruction_ids_sha256"] == "2a32b6cc09e5a48e87fff10b8a60803655826e17bac6f85540c9d7af357828a0"
assert selection_manifest["condition_counts"] == {"anomalous": 4220, "normal": 1780}
assert selection_manifest["task_condition_counts"] == selection_manifest["task_condition_quotas"]
assert selection_manifest["leakage"] == {"validation_image_overlap": 0, "test_image_overlap": 0}
assert min(value for counts in selection_manifest["task_category_counts"].values() for value in counts.values()) >= 10

In [ ]:
#@title 9. Configure persistence and verify conservative policy
training = yaml.safe_load(TRAINING_CONFIG.read_text(encoding="utf-8"))
training["checkpoints"]["persistent_output_dir"] = str(DRIVE_ROOT / "checkpoints" / BALANCED_RUN_ID)
training["checkpoints"]["sync_every_save"] = True
TRAINING_CONFIG.write_text(yaml.safe_dump(training, sort_keys=False), encoding="utf-8")
assert training["initial_adapter_path"] == f"outputs/training/{PILOT_RUN_ID}/final_adapter"
assert training["training"]["learning_rate"] == 1e-5
assert training["training"]["max_steps"] == 150
assert training["training"]["eval_steps"] == 25
adapter = PROJECT_ROOT / training["initial_adapter_path"]
assert (adapter / "adapter_config.json").is_file() and (adapter / "adapter_model.safetensors").is_file()
checkpoint_root = DRIVE_ROOT / "checkpoints" / BALANCED_RUN_ID
print("Existing Phase 11b checkpoints:", [p.name for p in sorted(checkpoint_root.glob("checkpoint-*"))] if checkpoint_root.is_dir() else [])
subprocess.run(["uv", "run", "visionassist", "training-environment", "--config", str(TRAINING_CONFIG)], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 10. One-batch Phase 11b smoke test
subprocess.run(["uv", "run", "visionassist", "training-smoke-test", "--config", str(TRAINING_CONFIG)], cwd=PROJECT_ROOT, check=True)
smoke_path = PROJECT_ROOT / "outputs/training" / BALANCED_RUN_ID / "one_batch_smoke_test.json"
smoke = json.loads(smoke_path.read_text(encoding="utf-8"))
print(json.dumps(smoke, indent=2))
assert smoke["passed"] and smoke["finite_gradients"] and smoke["nonzero_gradients"]

## C. Explicit short-training gate

Open this gate only after the exact fingerprint, condition quotas, zero leakage, empty/owned checkpoint directory, and smoke report pass.

In [ ]:
#@title 11. Start or resume the 150-step Phase 11b run
START_BALANCED_REPLAY = False
if not START_BALANCED_REPLAY: raise RuntimeError("Training gate is closed. Review Cells 8–10 first.")
subprocess.run(["uv", "run", "visionassist", "train-qlora", "--config", str(TRAINING_CONFIG), "--resume", "latest"], cwd=PROJECT_ROOT, check=True)

In [ ]:
#@title 12. Verify and persist the completed Phase 11b adapter
balanced_run = PROJECT_ROOT / "outputs/training" / BALANCED_RUN_ID
run_manifest = json.loads((balanced_run / "run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(run_manifest, indent=2))
assert run_manifest["status"] == "completed"
assert run_manifest["initial_adapter_path"] == training["initial_adapter_path"]
assert run_manifest["global_step"] == 150
assert (balanced_run / "final_adapter/adapter_model.safetensors").is_file()
drive_balanced = DRIVE_ROOT / "outputs/training" / BALANCED_RUN_ID
shutil.copytree(balanced_run, drive_balanced, dirs_exist_ok=True)
print("Saved Phase 11b artifacts to:", drive_balanced)

## D. Validation-only promotion gate

In [ ]:
#@title 13. Create resumable validation configuration
base_dir = PROJECT_ROOT / "configs/inference"
validation_config = yaml.safe_load((base_dir / "qwen25vl3b_overfit_checkpoint50_validation.yaml").read_text(encoding="utf-8"))
output = "outputs/post_training/qwen25vl3b_balanced_replay_best/validation"
validation_config.update({"run_id": "qwen25vl3b_balanced_replay_best_validation_v1", "adapter_path": f"outputs/training/{BALANCED_RUN_ID}/final_adapter", "output_dir": output, "partial_predictions_path": f"{output}/predictions.partial.jsonl", "predictions_path": f"{output}/predictions.jsonl", "errors_path": f"{output}/inference_errors.jsonl", "run_manifest_path": f"{output}/run_manifest.json", "evaluation_records_path": f"{output}/evaluation_records.jsonl", "benchmark_manifest_path": f"outputs/training/{BALANCED_RUN_ID}/dataset_manifest.json", "subset_limit": 1000, "subset_seed": 43, "overwrite": False, "persistent_output_dir": str(DRIVE_ROOT / "inference/qwen25vl3b_balanced_replay_best_validation"), "persistent_sync_every": 25})
VALIDATION_CONFIG = base_dir / "qwen25vl3b_balanced_replay_best_validation.yaml"
VALIDATION_CONFIG.write_text(yaml.safe_dump(validation_config, sort_keys=False), encoding="utf-8")
print(VALIDATION_CONFIG)

In [ ]:
#@title 14. Run Phase 11b validation
RUN_VALIDATION = False
if not RUN_VALIDATION: raise RuntimeError("Validation gate is closed.")
subprocess.run(["uv", "run", "visionassist", "evaluate-adapter", "--config", str(VALIDATION_CONFIG)], cwd=PROJECT_ROOT, check=True)
torch.cuda.empty_cache()

In [ ]:
#@title 15. Compare Phase 11b with promoted Phase 10
new_dir = PROJECT_ROOT / "outputs/post_training/qwen25vl3b_balanced_replay_best/validation"
new_metrics = json.loads((new_dir / "evaluation/metrics.json").read_text(encoding="utf-8"))
old_metrics = json.loads((LOCAL_PILOT_EVAL / "validation/evaluation/metrics.json").read_text(encoding="utf-8"))
for label, metrics in (("PHASE 10", old_metrics), ("PHASE 11B", new_metrics)):
    print(f"\n===== {label} VALIDATION ====="); print("failure_rate", metrics["failure_rate"]); print("failure_tags", metrics.get("failure_tag_counts", {}))
    for task, values in metrics["per_task"].items(): print(task, {key: value for key, value in values.items() if key not in {"per_label", "confusion_matrix"}})
drive_eval = DRIVE_ROOT / "outputs/post_training/qwen25vl3b_balanced_replay_best"
shutil.copytree(new_dir.parent, drive_eval, dirs_exist_ok=True)
print("Saved validation artifacts to:", drive_eval)

## Stop for review

Do not run the frozen test from this notebook. Share the complete Phase 10 versus Phase 11b validation comparison first. A separate test configuration should be created only after the validation promotion gates pass. Training checkpoints are under `checkpoints/qwen25vl3b_qlora_balanced_replay_v1`; inference partials synchronize every 25 records.